<a href="https://colab.research.google.com/github/SAIF-BT/SMIT/blob/main/SMIT_Assigment_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [2]:
df=pd.read_csv("/content/drive/MyDrive/healthcare-dataset-stroke-data.csv")

In [3]:
# intial anayaliss
print(df.head())
print("Data Info:")
print(df.info())
print("Missing values:")
print(df.isnull().sum())
print("Value counts for categorical columns:")
cat_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in cat_cols:
    print(f"\n{col}:\n{df[col].value_counts()}")

      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  
Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIn

In [4]:
# Check stroke distribution
print("Stroke distribution:")
print(df['stroke'].value_counts(normalize=True))

Stroke distribution:
stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64


In [5]:
# ANN Developmenr process
# Libaries required
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [6]:
# 1. Clean data
df_clean = df[df['gender'] != 'Other'].copy()
df_clean.drop(columns=['id'], inplace=True)

In [7]:
# 2. Define features and target
X = df_clean.drop(columns=['stroke'])
y = df_clean['stroke']

In [8]:
numeric_features = ['age','avg_glucose_level','bmi']
categorical_features = ['gender','ever_married','work_type','Residence_type','smoking_status','hypertension','heart_disease']
# hypertension and heart_disease are binary (0/1), but often treated as numeric or categorical.
# Since they are already 0/1, I'll keep them as is.

In [9]:
numeric_cols = ['age','avg_glucose_level','bmi']
categorical_cols = ['gender','ever_married','work_type','Residence_type','smoking_status']

In [10]:
# Create Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('onehot',OneHotEncoder(handle_unknown='ignore'))])

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)],
    remainder='passthrough')

In [12]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
#Define the ANN Model
# hidden_layer_sizes: (64, 32) is a decent starting point for this size dataset
ann_model = MLPClassifier(hidden_layer_sizes=(64, 32),activation='relu',solver='adam',max_iter=1000,random_state=42)



In [15]:
# Create full pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor),
 ('classifier', ann_model)])

In [16]:
clf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'avg_glucose_level',
                                                   'bmi']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'ever_married',
                                                   'work_type',
                                                   'Residence_type',
                                                   'smoking_status'])])),
                ('classifier',
                 MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000,
                               random_state=42))])

In [17]:
y_pred = clf.predict(X_test)

In [18]:
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-Score": f1_score(y_test, y_pred, zero_division=0)}
print("Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Evaluation Metrics:
Accuracy: 0.9325
Precision: 0.1935
Recall: 0.1200
F1-Score: 0.1481

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       972
           1       0.19      0.12      0.15        50

    accuracy                           0.93      1022
   macro avg       0.57      0.55      0.56      1022
weighted avg       0.92      0.93      0.92      1022



In [ ]:
#Accuracy is high (93.25%), this is primarily because the model is very good at predicting "No Stroke"(the majority class).
#The lower Precision and Recall for the "Stroke" class indicate that the model currently struggles to correctly identify positive stroke cases due to the significant class imbalance.
#In a real-world scenario, further techniques like SMOTE (oversampling) or adjusting class weights would be applied to improve the Recall for high-risk patients.